In [19]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

# Decision Tree Model from Scratch

In [20]:
class DecisionNode:
  def __init__(self, impurity=None, feature_index=None, threshold=None, left=None, right=None):
    self.left = left
    self.right = right
    # The largest impurity value of this node
    self.impurity = impurity
    # Index of the feature which make the best fit for this node.
    self.feature_index = feature_index
    # The threshold value for that feature to make the split.
    self.threshold = threshold

class LeafNode:
  def __init__(self, value):
    self.prediction_value = value

class DecisionTreeClassifierFromScratch:
  def __init__(self, min_sample_split=3, min_impurity=1e-7, max_depth=10, criterion='gini'):
    self.root = None
    self.min_sample_split = min_sample_split
    self.min_impurity = min_impurity
    self.max_depth = max_depth
    self.impurity_function = self._calculate_information_gain
    if criterion == 'entropy':
      self.criterion = self._entropy
      self.criterion_name = criterion
    else:
      self.criterion = self._gini_index
      self.criterion_name = 'gini'

  def _gini_index(self, y):
    gini = 1
    unique_value = np.unique(y)
    for val in unique_value:
      # probability of that class.
      p = np.sum(y == val) / len(y)
      gini += -np.square(p)
    return gini

  def _entropy(self, y):
    entropy = 0
    unique_value = np.unique(y)
    for val in unique_value:
      # probability of that class.
      p = np.sum(y == val) / len(y)
      entropy += -p * np.log2(p)
    return entropy

  def _calculate_information_gain(self, y, y1, y2):
    # :param y: target value.
    # :param y1: target value for dataset in the true split/right branch.
    # :param y2: target value for dataset in the false split/left branch.

    # propobility of true values.
    p = len(y1) / len(y)
    info_gain = self.criterion(y) - p * self.criterion(y1) - (1 - p) * self.criterion(y2)
    return info_gain

  def _leaf_value_calculation(self, y):
    most_frequent_label = None
    max_count = 0
    unique_labels = np.unique(y)
    # iterate over all the unique values and find their frequentcy count.
    for label in unique_labels:
      count = len( y[y == label])
      if count > max_count:
        most_frequent_label = label
        max_count = count
    return most_frequent_label

  def _partition_dataset(self, Xy, feature_index, threshold):
    col = Xy[:, feature_index]
    X_1 = Xy[col >= threshold]
    X_2 = Xy[col < threshold]

    return X_1, X_2

  def _find_best_split(self, Xy):
    best_question = tuple()
    best_datasplit = {}
    largest_impurity = 0
    n_features = (Xy.shape[1] - 1)
    # iterate over all the features.
    for feature_index in range(n_features):
      # find the unique values in that feature.
      unique_value = set(s for s in Xy[:,feature_index])
      # iterate over all the unique values to find the impurity.
      for threshold in unique_value:
        # split the dataset based on the feature value.
        true_xy, false_xy = self._partition_dataset(Xy, feature_index, threshold)

        # skip the node which has any on type 0. because this means it is already pure.
        if len(true_xy) > 0 and len(false_xy) > 0:
          # find the y values.
          y = Xy[:, -1]
          true_y = true_xy[:, -1]
          false_y = false_xy[:, -1]
          # calculate the impurity function.
          impurity = self.impurity_function(y, true_y, false_y)

          # if the calculated impurity is larger than save this value for comaparison (highest gain).
          if impurity > largest_impurity:
            largest_impurity = impurity
            best_question = (feature_index, threshold)
            best_datasplit = {
              "leftX": true_xy[:, :n_features],   # X of left subtree
              "lefty": true_xy[:, n_features:],   # y of left subtree
              "rightX": false_xy[:, :n_features],  # X of right subtree
              "righty": false_xy[:, n_features:]   # y of right subtree
            }

    return largest_impurity, best_question, best_datasplit

  def _build_tree(self, X, y, current_depth=0):
    n_samples , n_features = X.shape
    # Add y as last column of X
    Xy = np.column_stack((X, y))
    # find the Information gain on each feature each values and return the question which splits the data very well
    if (n_samples >= self.min_sample_split) and (current_depth < self.max_depth):
      # find the best split/ which question split the data well.
      impurity, question, best_datasplit = self._find_best_split(Xy)
      if impurity > self.min_impurity:
        # Build subtrees for the right and left branch.
        true_branch = self._build_tree(best_datasplit["leftX"], best_datasplit["lefty"], current_depth + 1)
        false_branch = self._build_tree(best_datasplit["rightX"], best_datasplit["righty"], current_depth + 1)
        return DecisionNode(impurity=impurity, feature_index=question[0], threshold=question[1],
                            left=true_branch, right=false_branch)

    leaf_value = self._leaf_value_calculation(y)
    return LeafNode(value=leaf_value)

  def fit(self, X, y):
    self.root = self._build_tree(X, y, current_depth=0)

  def predict_sample(self, x, tree=None):
    if isinstance(tree , LeafNode):
      return tree.prediction_value

    if tree is None:
      tree = self.root
    feature_value = x[tree.feature_index]
    branch = tree.right

    if isinstance(feature_value, int) or isinstance(feature_value, float):
      if feature_value >= tree.threshold:
        branch = tree.left
    elif feature_value == tree.threshold:
      branch = tree.left

    return self.predict_sample(x, branch)

  def predict(self, test_X):
    x = np.array(test_X)
    y_pred = [self.predict_sample(sample) for sample in x]
    y_pred = np.array(y_pred)
    return y_pred

  def draw_tree(self):
    self._draw_tree(self.root)

  def _draw_tree(self, tree = None, indentation = " ", depth=0):
    if isinstance(tree , LeafNode):
      print(indentation,"The predicted value -->", tree.prediction_value)
      return
    else:
      print(indentation,f"({depth}) Is {tree.feature_index}>={tree.threshold}?"
            f": {self.criterion_name}:{tree.impurity:.2f}")
      if tree.left is not None:
          print (indentation + '----- True branch :)')
          self._draw_tree(tree.left, indentation + "  ", depth+1)
      if tree.right is not None:
          print (indentation + '----- False branch :)')
          self._draw_tree(tree.right, indentation + "  ", depth+1)

# Attrition Prediction using Decision Tree from Scratch

In [ ]:
# Load the SMOTE-balanced dataset for attrition prediction
df = pd.read_csv('../data/processed_data_attrition.csv')
print("Dataset shape:", df.shape)

# Separate features and target
# Convert back to binary labels (0 and 1)
y_raw = df['Attrition'].values
X = df.drop(columns=['Attrition']).values
y = (y_raw == y_raw.max()).astype(int)

print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution:\n  No Attrition (0): {np.sum(y == 0)}\n  Attrition (1): {np.sum(y == 1)}")

Dataset shape: (2464, 37)

Columns:
['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'PerformanceIndex', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyAchievement', 'NumCompaniesWorked', 'OverTime', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'RemoteWork', 'MonthlyIncome', 'WelfareBenefits', 'InHouseFacility', 'ExternalFacility', 'ExtendedLeave', 'FlexibleWork', 'StressSelfReported', 'StressRating']

Features shape: (2464, 36)
Target distribution:
  No Attrition (0): 1232
  Attrition (1): 1232


In [ ]:
# Train/Test Split (80/20) 
from collections import Counter

def train_test_split_manual(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n = len(y)
    indices = np.arange(n)
    np.random.shuffle(indices)
    split = int(n * (1 - test_size))
    train_idx, test_idx = indices[:split], indices[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_manual(X, y, test_size=0.2, random_state=42)

Training set: 1971 samples
Test set:     493 samples

Training target distribution: {np.int64(0): np.int64(1000), np.int64(1): np.int64(971)}
Test target distribution:     {np.int64(0): np.int64(232), np.int64(1): np.int64(261)}


In [ ]:
# Train Decision Tree from Scratch (Gini criterion)
dt_gini = DecisionTreeClassifierFromScratch(min_sample_split=3, min_impurity=1e-7, max_depth=10, criterion='gini')
dt_gini.fit(X_train, y_train)

# Train Decision Tree from Scratch (Entropy criterion)
dt_entropy = DecisionTreeClassifierFromScratch(min_sample_split=3, min_impurity=1e-7, max_depth=10, criterion='entropy')
dt_entropy.fit(X_train, y_train)

Decision Tree (Gini) trained successfully!
Decision Tree (Entropy) trained successfully!


In [ ]:
# Evaluate both models
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    
    # Accuracy
    accuracy = np.sum(y_pred == y_test) / len(y_test)
    
    # Confusion Matrix 
    tp = np.sum((y_pred == 1) & (y_test == 1))
    tn = np.sum((y_pred == 0) & (y_test == 0))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))
    
    # Precision, Recall, F1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"                Predicted")
    print(f"                No(0)  Yes(1)")
    print(f"  Actual No(0)  {tn:<6} {fp}")
    print(f"  Actual Yes(1) {fn:<6} {tp}")
    print()
    
    return y_pred, accuracy

y_pred_gini, acc_gini = evaluate_model(dt_gini, X_test, y_test, "Decision Tree (Gini)")
y_pred_entropy, acc_entropy = evaluate_model(dt_entropy, X_test, y_test, "Decision Tree (Entropy)")

  Decision Tree (Gini)
  Accuracy:  0.8499
  Precision: 0.8450
  Recall:    0.8774
  F1-Score:  0.8609

  Confusion Matrix:
                Predicted
                No(0)  Yes(1)
  Actual No(0)  190    42
  Actual Yes(1) 32     229

  Decision Tree (Entropy)
  Accuracy:  0.8600
  Precision: 0.8636
  Recall:    0.8736
  F1-Score:  0.8686

  Confusion Matrix:
                Predicted
                No(0)  Yes(1)
  Actual No(0)  196    36
  Actual Yes(1) 33     228



In [ ]:
# Visualize the trained decision tree (Gini)
print("Decision Tree Structure (Gini criterion):")
print("(Feature indices map to columns in this order)")
feature_names = df.drop(columns=['Attrition']).columns.tolist()
for i, name in enumerate(feature_names):
    print(f"  Feature {i}: {name}")
print()
dt_gini.draw_tree()

Decision Tree Structure (Gini criterion):
(Feature indices map to columns in this order)
  Feature 0: Age
  Feature 1: BusinessTravel
  Feature 2: Department
  Feature 3: DistanceFromHome
  Feature 4: Education
  Feature 5: EducationField
  Feature 6: EnvironmentSatisfaction
  Feature 7: Gender
  Feature 8: PerformanceIndex
  Feature 9: JobInvolvement
  Feature 10: JobLevel
  Feature 11: JobRole
  Feature 12: JobSatisfaction
  Feature 13: MaritalStatus
  Feature 14: MonthlyAchievement
  Feature 15: NumCompaniesWorked
  Feature 16: OverTime
  Feature 17: PerformanceRating
  Feature 18: RelationshipSatisfaction
  Feature 19: StockOptionLevel
  Feature 20: TotalWorkingYears
  Feature 21: TrainingTimesLastYear
  Feature 22: WorkLifeBalance
  Feature 23: YearsAtCompany
  Feature 24: YearsInCurrentRole
  Feature 25: YearsSinceLastPromotion
  Feature 26: YearsWithCurrManager
  Feature 27: RemoteWork
  Feature 28: MonthlyIncome
  Feature 29: WelfareBenefits
  Feature 30: InHouseFacility
  Feat

In [26]:
# Hyperparameter tuning: try different max_depth values
print("Hyperparameter Tuning - Varying max_depth (Gini criterion)")
print(f"{'Max Depth':<12} {'Train Acc':<12} {'Test Acc':<12}")
print("-" * 36)

results = []
for depth in [3, 5, 7, 10, 15, 20]: 
    dt_temp = DecisionTreeClassifierFromScratch(min_sample_split=3, min_impurity=1e-7, max_depth=depth, criterion='gini')
    dt_temp.fit(X_train, y_train)
    
    train_pred = dt_temp.predict(X_train)
    test_pred = dt_temp.predict(X_test)
    
    train_acc = np.sum(train_pred == y_train) / len(y_train)
    test_acc = np.sum(test_pred == y_test) / len(y_test)
    
    results.append((depth, train_acc, test_acc))
    print(f"{depth:<12} {train_acc:<12.4f} {test_acc:<12.4f}")

# Find the best depth
best_result = max(results, key=lambda x: x[2])
print(f"\nBest max_depth: {best_result[0]} with test accuracy: {best_result[2]:.4f}")

Hyperparameter Tuning - Varying max_depth (Gini criterion)
Max Depth    Train Acc    Test Acc    
------------------------------------
3            0.7712       0.7505      
5            0.8417       0.7972      
7            0.9168       0.8377      
10           0.9807       0.8499      
15           0.9964       0.8418      
20           0.9964       0.8418      

Best max_depth: 10 with test accuracy: 0.8499


In [27]:
# Compare with sklearn DecisionTreeClassifier as baseline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sklearn_dt = DecisionTreeClassifier(max_depth=10, random_state=42)
sklearn_dt.fit(X_train, y_train)
y_pred_sklearn = sklearn_dt.predict(X_test)

print("=" * 50)
print("  Sklearn Decision Tree (Baseline)")
print("=" * 50)
print(f"  Accuracy: {accuracy_score(y_test, y_pred_sklearn):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_sklearn, target_names=['No Attrition', 'Attrition']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_sklearn))

print("\n" + "=" * 50)
print("  Summary Comparison")
print("=" * 50)
print(f"  From-Scratch (Gini):    {acc_gini:.4f}")
print(f"  From-Scratch (Entropy): {acc_entropy:.4f}")
print(f"  Sklearn Baseline:       {accuracy_score(y_test, y_pred_sklearn):.4f}")

  Sklearn Decision Tree (Baseline)
  Accuracy: 0.8641

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.87      0.84      0.85       232
   Attrition       0.86      0.89      0.87       261

    accuracy                           0.86       493
   macro avg       0.86      0.86      0.86       493
weighted avg       0.86      0.86      0.86       493

Confusion Matrix:
[[194  38]
 [ 29 232]]

  Summary Comparison
  From-Scratch (Gini):    0.8499
  From-Scratch (Entropy): 0.8600
  Sklearn Baseline:       0.8641
